## Part A - Application code
---
## ⚙️ Step 1: Imports & Configuration  *(`config.py`)*

In [ ]:
# =====================================================================
# STEP 1 - IMPORTS & CONFIGURATION                   (config.py)
# =====================================================================

import os
import sys

import numpy as np
import pandas as pd

# A notebook has no __file__, so everything is relative to the working folder
PACKAGE_DIR = os.getcwd()

# Data directory paths
DATA_DIR = os.path.join(PACKAGE_DIR, "data")
DEFAULT_DATA_FILE = os.path.join(DATA_DIR, "supermarket_sales.csv")

# Output directory and export paths
OUTPUT_DIR = os.path.join(DATA_DIR, "output")
MONTHLY_REPORT_CSV = os.path.join(OUTPUT_DIR, "monthly_report.csv")
REPORT_TXT = os.path.join(OUTPUT_DIR, "report.txt")

# Business Logic Defaults
DEFAULT_LOW_SALES_LIMIT = 5000.0

# Label shown when the built-in demo dataset is active
DEMO_SOURCE_LABEL = "built-in demo data (synthetic)"

# Fixed Domain Categories (Tuples)
BRANCHES = ("A", "B", "C")
PAYMENTS = ("Cash", "Credit Card", "Ewallet")
PRODUCT_LINES = (
    "Health And Beauty",
    "Electronic Accessories",
    "Home And Lifestyle",
    "Sports And Travel",
    "Food And Beverages",
    "Fashion Accessories",
)

# Expected CSV Column Schema (Tuple)
REQUIRED_COLUMNS = (
    "Invoice ID",
    "Branch",
    "City",
    "Customer type",
    "Gender",
    "Product line",
    "Unit price",
    "Quantity",
    "Tax 5%",
    "Total",
    "Date",
    "Time",
    "Payment",
    "cogs",
    "gross margin percentage",
    "gross income",
    "Rating",
)

# Text columns to strip and title-case
TEXT_COLUMNS = (
    "Branch",
    "City",
    "Customer type",
    "Gender",
    "Product line",
    "Payment",
)

# Numeric columns to ensure correct data types
NUMERIC_COLUMNS = (
    "Unit price",
    "Quantity",
    "Tax 5%",
    "Total",
    "cogs",
    "gross margin percentage",
    "gross income",
    "Rating",
)

print("Python Version :", sys.version.split()[0])
print("Working Dir    :", PACKAGE_DIR)
print("Configuration loaded.")

## 💾 Step 2: Data Layer  *(`store.py`)*
Loading, cleaning (PRD F1/F2), health check, CSV/text export, plus a small synthetic demo-data generator used only when no CSV is found.

In [ ]:
# =====================================================================
# STEP 2 - DATA LAYER                                  (store.py)
# Handles file I/O, data loading, cleaning, validation, and exports.
# Syllabus topics used: files, strings, dictionaries, tuples, lists, Pandas.
# Layer rule: Does NOT print to console. Does NOT call analytics or UI layers.
# =====================================================================


def load_data(file_path: str = None) -> pd.DataFrame:
    """
    Load supermarket sales CSV file into a pandas DataFrame.

    Parameters:
        file_path (str): Path to CSV data file (defaults to DEFAULT_DATA_FILE).

    Returns:
        pd.DataFrame: Loaded raw dataset.

    Raises:
        FileNotFoundError: If the specified file does not exist.
    """
    if file_path is None:
        file_path = DEFAULT_DATA_FILE

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset file not found at: {file_path}")

    return pd.read_csv(file_path)


def generate_demo_data(rows: int = 1000, seed: int = 42) -> pd.DataFrame:
    """
    Build a SYNTHETIC sample dataset with the same columns as supermarket_sales.csv.

    Used only when the real CSV is not available, so the app can still be demonstrated.
    A few duplicate invoices, missing ratings and messy text values are included on
    purpose, so the data-health check and cleaning steps have something to show.

    Parameters:
        rows (int): Number of base transactions to generate.
        seed (int): Random seed (same seed -> same data every time).

    Returns:
        pd.DataFrame: Raw (uncleaned) demo dataset.
    """
    rng = np.random.default_rng(seed)

    city_of_branch = {"A": "Yangon", "B": "Mandalay", "C": "Naypyitaw"}
    product_names = [p.capitalize() for p in PRODUCT_LINES]  # e.g. 'Health and beauty'
    payment_names = ["Ewallet", "Cash", "Credit card"]

    branch = rng.choice(list(BRANCHES), size=rows)
    unit_price = np.round(rng.uniform(10, 100, size=rows), 2)
    quantity = rng.integers(1, 11, size=rows)
    cogs = np.round(unit_price * quantity, 2)
    tax = np.round(cogs * 0.05, 4)

    days = pd.Timestamp("2019-01-01") + pd.to_timedelta(rng.integers(0, 90, size=rows), unit="D")
    hours = rng.integers(10, 21, size=rows)
    minutes = rng.integers(0, 60, size=rows)

    df = pd.DataFrame({
        "Invoice ID": [f"{100 + i % 900}-{10 + (i * 7) % 90}-{1000 + i}" for i in range(rows)],
        "Branch": branch,
        "City": [city_of_branch[str(b)] for b in branch],
        "Customer type": rng.choice(["Member", "Normal"], size=rows),
        "Gender": rng.choice(["Female", "Male"], size=rows),
        "Product line": rng.choice(product_names, size=rows),
        "Unit price": unit_price,
        "Quantity": quantity,
        "Tax 5%": tax,
        "Total": np.round(cogs + tax, 4),
        "Date": [f"{d.month}/{d.day}/{d.year}" for d in days],
        "Time": [f"{h}:{m:02d}" for h, m in zip(hours, minutes)],
        "Payment": rng.choice(payment_names, size=rows),
        "cogs": cogs,
        "gross margin percentage": 4.761904762,
        "gross income": tax,
        "Rating": np.round(rng.uniform(4, 10, size=rows), 1),
    })

    # Deliberate mess for the cleaning / health-check demo
    df.loc[df.sample(5, random_state=seed).index, "Rating"] = np.nan
    messy_idx = df.index[:6]
    df.loc[messy_idx, "Product line"] = ["  " + str(p).upper() + " " for p in df.loc[messy_idx, "Product line"]]
    duplicates = df.sample(3, random_state=seed + 1)

    return pd.concat([df, duplicates], ignore_index=True)


def clean_data(df: pd.DataFrame) -> tuple:
    """
    Clean and normalize sales data according to PRD F1 and F2:
    - Drops duplicate invoices by 'Invoice ID'
    - Strips whitespace and title-cases text fields
    - Safely converts numeric columns (preserving NaNs where values are missing)
    - Parses dates to derive 'Month' (month name)
    - Parses time to derive 'Hour' (0-23 integer)
    - Compiles data health check metrics

    Parameters:
        df (pd.DataFrame): Raw sales DataFrame.

    Returns:
        tuple: (cleaned_df: pd.DataFrame, health_stats: dict)
    """
    initial_rows = len(df)

    # 1. Drop duplicate invoices
    if "Invoice ID" in df.columns:
        df_cleaned = df.drop_duplicates(subset=["Invoice ID"]).copy()
    else:
        df_cleaned = df.copy()

    duplicates_removed = initial_rows - len(df_cleaned)

    # 2. Strip whitespace and title-case text columns
    for col in TEXT_COLUMNS:
        if col in df_cleaned.columns:
            # Preserve NaN values without converting them into the string 'Nan'
            non_null_mask = df_cleaned[col].notnull()
            df_cleaned.loc[non_null_mask, col] = (
                df_cleaned.loc[non_null_mask, col]
                .astype(str)
                .str.strip()
                .str.title()
            )

    # 3. Ensure numeric columns have correct float/int types
    for col in NUMERIC_COLUMNS:
        if col in df_cleaned.columns:
            df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce")

    # 4. Parse Date and extract Month
    if "Date" in df_cleaned.columns:
        parsed_dates = pd.to_datetime(df_cleaned["Date"], errors="coerce")
        df_cleaned["Month"] = parsed_dates.dt.month_name()
    else:
        df_cleaned["Month"] = "Unknown"

    # 5. Parse Time and extract Hour
    if "Time" in df_cleaned.columns:
        # Some times may have seconds or missing parts, split or parse
        time_series = df_cleaned["Time"].astype(str).str.strip()
        parsed_hours = []
        for val in time_series:
            parts = val.split(":")
            if parts and parts[0].isdigit():
                parsed_hours.append(int(parts[0]))
            else:
                parsed_hours.append(0)
        df_cleaned["Hour"] = parsed_hours
    else:
        df_cleaned["Hour"] = 0

    # 6. Gather health check statistics (dictionary)
    missing_values = {}
    for col in df_cleaned.columns:
        missing_count = int(df_cleaned[col].isnull().sum())
        missing_values[col] = missing_count

    health_stats = {
        "initial_rows": initial_rows,
        "cleaned_rows": len(df_cleaned),
        "duplicates_removed": duplicates_removed,
        "missing_values": missing_values,
        "total_missing": sum(missing_values.values()),
    }

    return df_cleaned, health_stats


def get_health_check(df: pd.DataFrame, duplicates_removed: int = 0) -> dict:
    """
    Return data health statistics for a DataFrame.

    Parameters:
        df (pd.DataFrame): DataFrame to inspect.
        duplicates_removed (int): Number of duplicate records previously dropped.

    Returns:
        dict: Health check metrics.
    """
    missing_values = {col: int(df[col].isnull().sum()) for col in df.columns}
    return {
        "total_rows": len(df),
        "duplicates_removed": duplicates_removed,
        "missing_values": missing_values,
        "total_missing": sum(missing_values.values()),
    }


def save_csv(df: pd.DataFrame, file_path: str = MONTHLY_REPORT_CSV) -> bool:
    """
    Save DataFrame to CSV file, ensuring target directory exists.

    Parameters:
        df (pd.DataFrame): DataFrame to export.
        file_path (str): Output file destination.

    Returns:
        bool: True if file write succeeded.
    """
    out_dir = os.path.dirname(file_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    df.to_csv(file_path, index=False)
    return True


def save_text_report(content: str, file_path: str = REPORT_TXT) -> bool:
    """
    Write text content to file, creating destination folders if needed.

    Parameters:
        content (str): Text report content.
        file_path (str): File destination path.

    Returns:
        bool: True if write succeeded.
    """
    out_dir = os.path.dirname(file_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return True


print("Data layer loaded.")

## 📊 Step 3: Analytics Layer  *(`analytics.py`)*
Revenue views, rankings, payment share, ratings and NumPy bill statistics.

In [ ]:
# =====================================================================
# STEP 3 - ANALYTICS LAYER                         (analytics.py)
# Revenue aggregation, product rankings, payment shares, rating
# metrics, and NumPy statistical analysis.
# Layer rule: Does NOT print to console. Does NOT read files directly.
# =====================================================================


def get_total_revenue(df: pd.DataFrame) -> float:
    """
    Calculate the total gross revenue from all transactions.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        float: Total revenue sum.
    """
    if df.empty or "Total" not in df.columns:
        return 0.0
    return float(df["Total"].sum())


def get_total_units(df: pd.DataFrame) -> int:
    """
    Calculate the total quantity of items sold.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        int: Total units sold.
    """
    if df.empty or "Quantity" not in df.columns:
        return 0
    return int(df["Quantity"].sum())


def get_revenue_by_branch(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate sales metrics grouped by Branch.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        pd.DataFrame: Table sorted by Revenue descending with columns:
                      ['Branch', 'Revenue', 'Quantity', 'Transactions', 'Avg_Bill']
    """
    if df.empty:
        return pd.DataFrame(columns=["Branch", "Revenue", "Quantity", "Transactions", "Avg_Bill"])

    grouped = df.groupby("Branch", as_index=False).agg(
        Revenue=("Total", "sum"),
        Quantity=("Quantity", "sum"),
        Transactions=("Invoice ID", "count"),
        Avg_Bill=("Total", "mean"),
    )
    return grouped.sort_values(by="Revenue", ascending=False).reset_index(drop=True)


def get_revenue_by_product_line(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate sales metrics grouped by Product line.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        pd.DataFrame: Table sorted by Revenue descending with columns:
                      ['Product line', 'Revenue', 'Quantity', 'Transactions', 'Avg_Price']
    """
    if df.empty:
        return pd.DataFrame(columns=["Product line", "Revenue", "Quantity", "Transactions", "Avg_Price"])

    grouped = df.groupby("Product line", as_index=False).agg(
        Revenue=("Total", "sum"),
        Quantity=("Quantity", "sum"),
        Transactions=("Invoice ID", "count"),
        Avg_Price=("Unit price", "mean"),
    )
    return grouped.sort_values(by="Revenue", ascending=False).reset_index(drop=True)


def get_revenue_by_month(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate sales metrics grouped by Month.

    Parameters:
        df (pd.DataFrame): Sales DataFrame containing 'Month' column.

    Returns:
        pd.DataFrame: Table grouped by Month with columns:
                      ['Month', 'Revenue', 'Quantity', 'Transactions']
    """
    if df.empty or "Month" not in df.columns:
        return pd.DataFrame(columns=["Month", "Revenue", "Quantity", "Transactions"])

    grouped = df.groupby("Month", as_index=False).agg(
        Revenue=("Total", "sum"),
        Quantity=("Quantity", "sum"),
        Transactions=("Invoice ID", "count"),
    )

    # Order months chronologically if possible
    month_order = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December",
    ]
    grouped["_sort_key"] = grouped["Month"].apply(
        lambda m: month_order.index(m) if m in month_order else 99
    )
    grouped = grouped.sort_values(by="_sort_key").drop(columns=["_sort_key"]).reset_index(drop=True)
    return grouped


def get_revenue_by_hour(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate sales metrics grouped by Hour of the day.

    Parameters:
        df (pd.DataFrame): Sales DataFrame containing 'Hour' column.

    Returns:
        pd.DataFrame: Table grouped by Hour sorted chronologically.
    """
    if df.empty or "Hour" not in df.columns:
        return pd.DataFrame(columns=["Hour", "Revenue", "Quantity", "Transactions"])

    grouped = df.groupby("Hour", as_index=False).agg(
        Revenue=("Total", "sum"),
        Quantity=("Quantity", "sum"),
        Transactions=("Invoice ID", "count"),
    )
    return grouped.sort_values(by="Hour", ascending=True).reset_index(drop=True)


def get_best_and_slowest_products(df: pd.DataFrame) -> dict:
    """
    Determine the best and slowest selling products ranked by revenue and by units.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        dict: Detailed rankings and top/bottom items:
              {
                  'best_by_revenue': dict,
                  'slowest_by_revenue': dict,
                  'best_by_units': dict,
                  'slowest_by_units': dict,
                  'revenue_ranking': pd.DataFrame,
                  'units_ranking': pd.DataFrame
              }
    """
    prod_summary = get_revenue_by_product_line(df)
    if prod_summary.empty:
        empty_item = {"product_line": "N/A", "revenue": 0.0, "units": 0}
        return {
            "best_by_revenue": empty_item,
            "slowest_by_revenue": empty_item,
            "best_by_units": empty_item,
            "slowest_by_units": empty_item,
            "revenue_ranking": prod_summary,
            "units_ranking": prod_summary,
        }

    # Revenue ranking
    rev_sorted = prod_summary.sort_values(by="Revenue", ascending=False).reset_index(drop=True)
    units_sorted = prod_summary.sort_values(by="Quantity", ascending=False).reset_index(drop=True)

    best_rev_row = rev_sorted.iloc[0]
    slow_rev_row = rev_sorted.iloc[-1]
    best_units_row = units_sorted.iloc[0]
    slow_units_row = units_sorted.iloc[-1]

    return {
        "best_by_revenue": {
            "product_line": str(best_rev_row["Product line"]),
            "revenue": float(best_rev_row["Revenue"]),
            "units": int(best_rev_row["Quantity"]),
        },
        "slowest_by_revenue": {
            "product_line": str(slow_rev_row["Product line"]),
            "revenue": float(slow_rev_row["Revenue"]),
            "units": int(slow_rev_row["Quantity"]),
        },
        "best_by_units": {
            "product_line": str(best_units_row["Product line"]),
            "revenue": float(best_units_row["Revenue"]),
            "units": int(best_units_row["Quantity"]),
        },
        "slowest_by_units": {
            "product_line": str(slow_units_row["Product line"]),
            "revenue": float(slow_units_row["Revenue"]),
            "units": int(slow_units_row["Quantity"]),
        },
        "revenue_ranking": rev_sorted,
        "units_ranking": units_sorted,
    }


def get_payment_share(df: pd.DataFrame) -> dict:
    """
    Calculate the percentage share and volume for each payment method.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        dict: {
            'count_share_pct': dict[str, float],
            'revenue_share_pct': dict[str, float],
            'summary_table': pd.DataFrame
        }
    """
    if df.empty or "Payment" not in df.columns:
        return {"count_share_pct": {}, "revenue_share_pct": {}, "summary_table": pd.DataFrame()}

    total_rev = get_total_revenue(df)
    total_tx = len(df)

    summary = df.groupby("Payment", as_index=False).agg(
        Transactions=("Invoice ID", "count"),
        Revenue=("Total", "sum"),
    )

    summary["Tx_Share_Pct"] = (
        (summary["Transactions"] / total_tx * 100).round(2) if total_tx > 0 else 0.0
    )
    summary["Revenue_Share_Pct"] = (
        (summary["Revenue"] / total_rev * 100).round(2) if total_rev > 0 else 0.0
    )
    summary = summary.sort_values(by="Revenue", ascending=False).reset_index(drop=True)

    count_share = dict(zip(summary["Payment"], summary["Tx_Share_Pct"]))
    revenue_share = dict(zip(summary["Payment"], summary["Revenue_Share_Pct"]))

    return {
        "count_share_pct": count_share,
        "revenue_share_pct": revenue_share,
        "summary_table": summary,
    }


def get_rating_averages(df: pd.DataFrame) -> dict:
    """
    Calculate customer rating averages overall, by branch, and by product line.
    Handles NaN values safely without crashing or skewing.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        dict: {
            'overall_avg': float,
            'by_branch': dict[str, float],
            'by_product': dict[str, float]
        }
    """
    if df.empty or "Rating" not in df.columns:
        return {"overall_avg": 0.0, "by_branch": {}, "by_product": {}}

    valid_ratings = df["Rating"].dropna()
    overall = float(valid_ratings.mean()) if not valid_ratings.empty else 0.0

    branch_ratings = (
        df.groupby("Branch")["Rating"].mean().dropna().round(2).to_dict()
    )
    product_ratings = (
        df.groupby("Product line")["Rating"].mean().dropna().round(2).to_dict()
    )

    return {
        "overall_avg": round(overall, 2),
        "by_branch": branch_ratings,
        "by_product": product_ratings,
    }


def get_bill_statistics(df: pd.DataFrame) -> dict:
    """
    Compute invoice bill statistics using NumPy according to PRD F8:
    Mean, Median, Standard Deviation, and 90th Percentile.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        dict: NumPy statistical metrics on invoice totals.
    """
    if df.empty or "Total" not in df.columns:
        return {
            "mean": 0.0,
            "median": 0.0,
            "std_dev": 0.0,
            "percentile_90": 0.0,
            "min": 0.0,
            "max": 0.0,
            "count": 0,
        }

    # Extract NumPy array from Total column, excluding any nulls
    totals = np.array(df["Total"].dropna(), dtype=float)

    if len(totals) == 0:
        return {
            "mean": 0.0,
            "median": 0.0,
            "std_dev": 0.0,
            "percentile_90": 0.0,
            "min": 0.0,
            "max": 0.0,
            "count": 0,
        }

    return {
        "mean": float(np.mean(totals)),
        "median": float(np.median(totals)),
        "std_dev": float(np.std(totals)),
        "percentile_90": float(np.percentile(totals, 90)),
        "min": float(np.min(totals)),
        "max": float(np.max(totals)),
        "count": int(len(totals)),
    }


print("Analytics layer loaded.")

## ⚠️ Step 4: Alerts Layer  *(`alerts.py`)*
Low-sales velocity alerts and branch product gap analysis (set theory).

In [ ]:
# =====================================================================
# STEP 4 - ALERTS LAYER                              (alerts.py)
# Low-sales velocity alerts and branch product gap analysis.
# Syllabus topics used: dictionaries, sets, operators, lists, functions.
# Layer rule: Does NOT print to console. Does NOT read files directly.
# =====================================================================


def get_low_sales_alerts(df: pd.DataFrame, limit: float = DEFAULT_LOW_SALES_LIMIT) -> dict:
    """
    Detect product lines with revenue below a configurable threshold limit (PRD F6).
    Uses operators and conditional logic to categorize sales velocity.

    Parameters:
        df (pd.DataFrame): Cleaned sales DataFrame.
        limit (float): Revenue threshold for alert (runtime configurable).

    Returns:
        dict: {
            'threshold_limit': float,
            'total_products_checked': int,
            'alert_count': int,
            'alerts': list[dict],
            'safe_products': list[dict]
        }
    """
    if df.empty or "Product line" not in df.columns or "Total" not in df.columns:
        return {
            "threshold_limit": limit,
            "total_products_checked": 0,
            "alert_count": 0,
            "alerts": [],
            "safe_products": [],
        }

    grouped = df.groupby("Product line", as_index=False).agg(
        Revenue=("Total", "sum"),
        Quantity=("Quantity", "sum"),
        Transactions=("Invoice ID", "count"),
    )

    alerts = []
    safe_products = []

    for _, row in grouped.iterrows():
        prod = str(row["Product line"])
        rev = float(row["Revenue"])
        qty = int(row["Quantity"])
        tx_count = int(row["Transactions"])

        pct_reached = round((rev / limit * 100), 2) if limit > 0 else 100.0
        deficit = round(limit - rev, 2) if rev < limit else 0.0

        item_info = {
            "product_line": prod,
            "revenue": round(rev, 2),
            "units": qty,
            "transactions": tx_count,
            "threshold": limit,
            "deficit": deficit,
            "pct_reached": pct_reached,
        }

        # Operators: comparisons and boolean checks
        if rev < limit:
            # Determine specific advisory using operators
            if qty < 100:
                advice = "Low demand: Consider promotional discount or bundling."
            else:
                advice = "Moderate unit movement but low revenue: Review item pricing."
            item_info["status"] = "LOW_SALES_ALERT"
            item_info["recommendation"] = advice
            alerts.append(item_info)
        else:
            item_info["status"] = "HEALTHY"
            item_info["recommendation"] = "Sales meet target threshold."
            safe_products.append(item_info)

    # Sort alerts by deficit descending (worst first)
    alerts.sort(key=lambda x: x["deficit"], reverse=True)

    return {
        "threshold_limit": limit,
        "total_products_checked": len(grouped),
        "alert_count": len(alerts),
        "alerts": alerts,
        "safe_products": safe_products,
    }


def get_branch_product_gap(df: pd.DataFrame) -> dict:
    """
    Perform set difference and intersection operations across branches (PRD F7).
    Discovers:
    - Products sold in ALL branches (Set Intersection)
    - Products unique to a specific branch (Set Difference)
    - Catalog completeness across branches

    Parameters:
        df (pd.DataFrame): Cleaned sales DataFrame.

    Returns:
        dict: Set analysis containing sets and formatted summary lists.
    """
    if df.empty or "Branch" not in df.columns or "Product line" not in df.columns:
        return {
            "branch_sets": {},
            "common_all_branches": set(),
            "all_catalog_products": set(),
            "exclusive_by_branch": {},
            "pairwise_differences": [],
        }

    available_branches = sorted(df["Branch"].dropna().unique())

    # Build dictionary of sets: branch -> set of product lines
    branch_sets = {}
    for b in available_branches:
        branch_products = set(df[df["Branch"] == b]["Product line"].dropna().unique())
        branch_sets[b] = branch_products

    # 1. Set Union: All distinct products across the entire business
    all_catalog = set.union(*branch_sets.values()) if branch_sets else set()

    # 2. Set Intersection: Products present in every branch
    if branch_sets:
        common_all = set.intersection(*branch_sets.values())
    else:
        common_all = set()

    # 3. Set Difference: Products exclusive to each branch (in branch B but not in any other)
    exclusive_by_branch = {}
    for b, b_set in branch_sets.items():
        other_sets = [s for other_b, s in branch_sets.items() if other_b != b]
        if other_sets:
            others_union = set.union(*other_sets)
            exclusive_by_branch[b] = b_set - others_union
        else:
            exclusive_by_branch[b] = set()

    # 4. Pairwise Set Differences between branches
    pairwise = []
    for i, b1 in enumerate(available_branches):
        for b2 in available_branches[i + 1:]:
            s1 = branch_sets[b1]
            s2 = branch_sets[b2]
            only_in_first = s1 - s2
            only_in_second = s2 - s1
            shared = s1 & s2
            pairwise.append({
                "branches": (b1, b2),
                "only_in_first": only_in_first,
                "only_in_second": only_in_second,
                "shared": shared,
            })

    return {
        "branch_sets": branch_sets,
        "common_all_branches": common_all,
        "all_catalog_products": all_catalog,
        "exclusive_by_branch": exclusive_by_branch,
        "pairwise_differences": pairwise,
    }


print("Alerts layer loaded.")

## 🔍 Step 5: Search Layer  *(`search.py`)*
Case- and space-tolerant search.

In [ ]:
# =====================================================================
# STEP 5 - SEARCH LAYER                              (search.py)
# Case- and space-tolerant search across sales records.
# Syllabus topics used: strings, lists, dictionaries, functions, operators, Pandas.
# Layer rule: Does NOT print to console. Does NOT read files directly.
# =====================================================================


def normalize_string(text: str) -> str:
    """
    Clean and normalize user query by stripping whitespace and lowercasing.

    Parameters:
        text (str): Input text string.

    Returns:
        str: Cleaned lowercase string with collapsed whitespace.
    """
    if not isinstance(text, str):
        text = str(text) if text is not None else ""
    return " ".join(text.strip().lower().split())


def search_records(
    df: pd.DataFrame,
    query: str,
    search_field: str = "all",
) -> pd.DataFrame:
    """
    Search transactions by product line, branch, date, or invoice ID (PRD F9).
    Offers full tolerance for casing, leading/trailing spaces, and internal multi-spaces.

    Parameters:
        df (pd.DataFrame): Cleaned sales DataFrame.
        query (str): Keyword or phrase to look for.
        search_field (str): Field to search in ('all', 'product', 'branch', 'invoice', 'date', 'city').

    Returns:
        pd.DataFrame: Matching subset of sales records.
    """
    if df.empty or not query or not query.strip():
        return df.copy()

    clean_q = normalize_string(query)
    field_lower = search_field.strip().lower()

    # Define candidate columns based on user selection
    field_map = {
        "product": ["Product line"],
        "branch": ["Branch"],
        "invoice": ["Invoice ID"],
        "date": ["Date", "Month"],
        "city": ["City"],
        "payment": ["Payment"],
        "all": ["Invoice ID", "Branch", "City", "Product line", "Payment", "Date", "Month"],
    }

    target_cols = field_map.get(field_lower, field_map["all"])
    available_cols = [c for c in target_cols if c in df.columns]

    if not available_cols:
        return pd.DataFrame(columns=df.columns)

    # Build boolean mask across candidate columns using vectorized string operations
    mask = pd.Series(False, index=df.index)
    for col in available_cols:
        # Normalize column string representations on the fly
        col_normalized = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
        )
        mask = mask | col_normalized.str.contains(clean_q, regex=False, na=False)

    return df[mask].reset_index(drop=True)


def get_transaction_by_invoice(df: pd.DataFrame, invoice_id: str) -> dict:
    """
    Lookup a specific transaction by invoice ID with whitespace and case tolerance.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.
        invoice_id (str): Target invoice ID.

    Returns:
        dict: Transaction fields as key-value pairs, or empty dict if not found.
    """
    if df.empty or "Invoice ID" not in df.columns or not invoice_id:
        return {}

    target = normalize_string(invoice_id)
    matches = df[
        df["Invoice ID"].astype(str).str.strip().str.lower() == target
    ]

    if matches.empty:
        return {}

    return matches.iloc[0].to_dict()


def filter_records(
    df: pd.DataFrame,
    branch: str = None,
    product_line: str = None,
    min_total: float = None,
    max_total: float = None,
) -> pd.DataFrame:
    """
    Multi-criteria filter using operators and conditional combinations.

    Parameters:
        df (pd.DataFrame): Sales DataFrame.
        branch (str, optional): Filter by branch letter.
        product_line (str, optional): Filter by product line name.
        min_total (float, optional): Minimum invoice total.
        max_total (float, optional): Maximum invoice total.

    Returns:
        pd.DataFrame: Filtered DataFrame.
    """
    if df.empty:
        return df.copy()

    filtered = df.copy()

    if branch:
        filtered = filtered[
            filtered["Branch"].astype(str).str.strip().str.upper() == branch.strip().upper()
        ]

    if product_line:
        clean_prod = normalize_string(product_line)
        filtered = filtered[
            filtered["Product line"].astype(str).str.strip().str.lower() == clean_prod
        ]

    if min_total is not None and "Total" in filtered.columns:
        filtered = filtered[filtered["Total"] >= min_total]

    if max_total is not None and "Total" in filtered.columns:
        filtered = filtered[filtered["Total"] <= max_total]

    return filtered.reset_index(drop=True)


print("Search layer loaded.")

## 🧾 Step 6: Report Layer  *(`report.py`)*
ASCII tables, text bar charts and report export.

In [ ]:
# =====================================================================
# STEP 6 - REPORT LAYER                              (report.py)
# ASCII text bar charts, aligned console tables, and exportable reports.
# Syllabus topics used: strings, lists, dictionaries, files, functions, operators.
# Layer rule: Formats and prints data. Calls the data layer (Step 2) for file export.
# Universal ASCII design: safe across all terminal encodings (incl. Windows cp1252 & Colab).
# =====================================================================


def format_table(headers: list, rows: list, alignments: list = None) -> str:
    """
    Format data rows and headers into a clean, aligned plain-text table.

    Parameters:
        headers (list[str]): Column titles.
        rows (list[list]): List of row value lists.
        alignments (list[str], optional): 'L' for left, 'R' for right.

    Returns:
        str: Aligned multi-line table string.
    """
    if not headers:
        return ""

    str_headers = [str(h) for h in headers]
    str_rows = [[str(cell) for cell in row] for row in rows]

    # Calculate column widths
    col_widths = [len(h) for h in str_headers]
    for row in str_rows:
        for i, cell in enumerate(row):
            if i < len(col_widths):
                col_widths[i] = max(col_widths[i], len(cell))

    # Default alignments: right for numbers/currencies, left for text
    if not alignments:
        alignments = []
        for h in str_headers:
            h_lower = h.lower()
            if any(term in h_lower for term in ("rev", "total", "qty", "count", "stat", "pct", "%", "price", "bill", "unit", "deficit")):
                alignments.append("R")
            else:
                alignments.append("L")

    while len(alignments) < len(col_widths):
        alignments.append("L")

    sep_line = "+" + "+".join("-" * (w + 2) for w in col_widths) + "+"

    def format_row(cells):
        formatted_cells = []
        for i, cell in enumerate(cells):
            w = col_widths[i]
            align = alignments[i] if i < len(alignments) else "L"
            if align == "R":
                formatted_cells.append(f" {cell:>{w}} ")
            else:
                formatted_cells.append(f" {cell:<{w}} ")
        return "|" + "|".join(formatted_cells) + "|"

    output_lines = [sep_line, format_row(str_headers), sep_line]
    for row in str_rows:
        output_lines.append(format_row(row))
    output_lines.append(sep_line)

    return "\n".join(output_lines)


def build_text_bar(value: float, max_value: float, width: int = 28, char: str = "#") -> str:
    """
    Generate a text bar string proportional to value / max_value (PRD F10).
    Constructed using pure string multiplication with universal ASCII chars.

    Parameters:
        value (float): Metric value.
        max_value (float): Maximum scale value.
        width (int): Max bar character width.
        char (str): ASCII character used to draw the bar.

    Returns:
        str: String bar representation.
    """
    if max_value <= 0 or value <= 0:
        bar_len = 0
    else:
        bar_len = int((value / max_value) * width)
        if bar_len > width:
            bar_len = width
        elif bar_len == 0 and value > 0:
            bar_len = 1

    bar_str = char * bar_len
    empty_str = " " * (width - bar_len)
    return f"[{bar_str}{empty_str}]"


def render_bar_chart(title: str, items: list, unit: str = "$", width: int = 26) -> str:
    """
    Render a formatted visual text bar chart (PRD F10).

    Parameters:
        title (str): Chart heading.
        items (list[tuple]): List of (label: str, value: float).
        unit (str): Display prefix or suffix for numbers.
        width (int): Bar length limit.

    Returns:
        str: Rendered ASCII bar chart.
    """
    if not items:
        return f"{title}\n  (No data to display)"

    max_val = max(val for _, val in items) if items else 1.0
    max_label_len = max(len(str(lbl)) for lbl, _ in items) if items else 10

    lines = [
        f"=== {title} ===",
        "-" * (max_label_len + width + 24),
    ]

    for label, val in items:
        bar = build_text_bar(val, max_val, width=width)
        if unit == "$":
            val_str = f"${val:,.2f}"
        elif unit == "%":
            val_str = f"{val:.1f}%"
        else:
            val_str = f"{val:,.0f} {unit}"
        lines.append(f"  {label:<{max_label_len}} | {bar} {val_str:>12}")

    lines.append("-" * (max_label_len + width + 24))
    return "\n".join(lines)


def display_header(title: str, subtitle: str = ""):
    """Print an eye-catching section banner."""
    print()
    print("=" * 68)
    print(f"  {title.upper()}")
    if subtitle:
        print(f"  {subtitle}")
    print("=" * 68)


def display_health_check(health_stats: dict):
    """Display data quality and integrity health check (PRD F2)."""
    display_header("Data Health Check", "Integrity, Duplicates & Missing Values")
    print(f"  Total Clean Records       : {health_stats.get('cleaned_rows', 0):,}")
    print(f"  Duplicate Invoices Dropped: {health_stats.get('duplicates_removed', 0):,}")
    print(f"  Total Missing Fields      : {health_stats.get('total_missing', 0):,}")
    print()

    missing = health_stats.get("missing_values", {})
    headers = ["Column Name", "Missing Count", "Status"]
    rows = []
    for col, count in missing.items():
        status = "OK (Complete)" if count == 0 else f"Notice ({count} null)"
        rows.append([col, count, status])
    print(format_table(headers, rows))


def display_revenue_views(df: pd.DataFrame):
    """Display revenue aggregated by branch, product, month, and hour (PRD F3)."""
    display_header("Revenue Views", "Sales Aggregated by Dimension")

    # 1. By Branch
    print("\n--- 1. Revenue by Branch ---")
    b_df = get_revenue_by_branch(df)
    b_rows = [
        [r["Branch"], f"${r['Revenue']:,.2f}", f"{r['Quantity']:,}", f"{r['Transactions']:,}", f"${r['Avg_Bill']:,.2f}"]
        for _, r in b_df.iterrows()
    ]
    print(format_table(["Branch", "Revenue", "Units Sold", "Transactions", "Avg Bill"], b_rows))

    # 2. By Product Line
    print("\n--- 2. Revenue by Product Line ---")
    p_df = get_revenue_by_product_line(df)
    p_rows = [
        [r["Product line"], f"${r['Revenue']:,.2f}", f"{r['Quantity']:,}", f"{r['Transactions']:,}", f"${r['Avg_Price']:,.2f}"]
        for _, r in p_df.iterrows()
    ]
    print(format_table(["Product Line", "Revenue", "Units Sold", "Transactions", "Avg Unit Price"], p_rows))

    # 3. By Month
    print("\n--- 3. Revenue by Month ---")
    m_df = get_revenue_by_month(df)
    m_rows = [
        [r["Month"], f"${r['Revenue']:,.2f}", f"{r['Quantity']:,}", f"{r['Transactions']:,}"]
        for _, r in m_df.iterrows()
    ]
    print(format_table(["Month", "Revenue", "Units Sold", "Transactions"], m_rows))

    # 4. By Hour
    print("\n--- 4. Peak Hours Revenue ---")
    h_df = get_revenue_by_hour(df)
    h_rows = [
        [f"{int(r['Hour']):02d}:00", f"${r['Revenue']:,.2f}", f"{r['Quantity']:,}", f"{r['Transactions']:,}"]
        for _, r in h_df.iterrows()
    ]
    print(format_table(["Hour", "Revenue", "Units Sold", "Transactions"], h_rows))


def display_product_rankings(df: pd.DataFrame):
    """Display best and slowest products ranked by revenue and units (PRD F4)."""
    display_header("Product Performance Rankings", "Best & Slowest Performing Items")
    rankings = get_best_and_slowest_products(df)

    best_rev = rankings["best_by_revenue"]
    slow_rev = rankings["slowest_by_revenue"]
    best_units = rankings["best_by_units"]
    slow_units = rankings["slowest_by_units"]

    print(f"  [BEST] Highest Revenue Leader : {best_rev['product_line']} (${best_rev['revenue']:,.2f} from {best_rev['units']:,} units)")
    print(f"  [LOW]  Lowest Revenue Performer: {slow_rev['product_line']} (${slow_rev['revenue']:,.2f} from {slow_rev['units']:,} units)")
    print(f"  [BEST] Most Units Sold        : {best_units['product_line']} ({best_units['units']:,} units sold, ${best_units['revenue']:,.2f})")
    print(f"  [LOW]  Fewest Units Sold      : {slow_units['product_line']} ({slow_units['units']:,} units sold, ${slow_units['revenue']:,.2f})")
    print()

    print("Complete Product Ranking (Sorted by Revenue):")
    rev_df = rankings["revenue_ranking"]
    rows = [
        [i + 1, r["Product line"], f"${r['Revenue']:,.2f}", f"{r['Quantity']:,}", f"{r['Transactions']:,}"]
        for i, (_, r) in enumerate(rev_df.iterrows())
    ]
    print(format_table(["Rank", "Product Line", "Revenue", "Units Sold", "Transactions"], rows))


def display_payment_share(df: pd.DataFrame):
    """Display payment method percentage breakdown (PRD F5)."""
    display_header("Payment Method Breakdown", "Transaction Volume and Revenue Share")
    pay_data = get_payment_share(df)
    summary_df = pay_data.get("summary_table", pd.DataFrame())

    rows = [
        [
            r["Payment"],
            f"{r['Transactions']:,}",
            f"{r['Tx_Share_Pct']:.2f}%",
            f"${r['Revenue']:,.2f}",
            f"{r['Revenue_Share_Pct']:.2f}%",
        ]
        for _, r in summary_df.iterrows()
    ]
    print(format_table(["Payment Method", "Transactions", "Tx Share %", "Revenue", "Revenue Share %"], rows))

    items = [(str(r["Payment"]), float(r["Revenue_Share_Pct"])) for _, r in summary_df.iterrows()]
    print()
    print(render_bar_chart("Payment Revenue Share (%)", items, unit="%"))


def display_low_sales_alerts(df: pd.DataFrame, limit: float = DEFAULT_LOW_SALES_LIMIT):
    """Display products triggering low-sales alerts (PRD F6)."""
    display_header("Low-Sales Velocity Alerts", f"Target Threshold: ${limit:,.2f}")
    alerts_data = get_low_sales_alerts(df, limit)

    flagged = alerts_data["alerts"]
    safe = alerts_data["safe_products"]

    print(f"  Total Product Lines Checked : {alerts_data['total_products_checked']}")
    print(f"  Products Requiring Attention: {len(flagged)}")
    print()

    if not flagged:
        print(f"  [OK] Excellent: All product lines met or exceeded the ${limit:,.2f} threshold!")
    else:
        print("  [ALERT] FLAGGED ITEMS BELOW TARGET THRESHOLD:")
        headers = ["Product Line", "Revenue", "Target", "Deficit", "% Achieved", "Advisory"]
        rows = [
            [
                item["product_line"],
                f"${item['revenue']:,.2f}",
                f"${item['threshold']:,.2f}",
                f"${item['deficit']:,.2f}",
                f"{item['pct_reached']:.1f}%",
                item["recommendation"],
            ]
            for item in flagged
        ]
        print(format_table(headers, rows))

    if safe:
        print(f"\n  [HEALTHY] Products Above ${limit:,.2f}:")
        for s in safe:
            print(f"      - {s['product_line']}: ${s['revenue']:,.2f} ({s['pct_reached']:.1f}% of target)")


def display_branch_gap(df: pd.DataFrame):
    """Display branch product gap set analysis (PRD F7)."""
    display_header("Branch Product Gap Analysis", "Set Intersection & Differences")
    gap_data = get_branch_product_gap(df)

    branch_sets = gap_data["branch_sets"]
    common_all = gap_data["common_all_branches"]
    all_cat = gap_data["all_catalog_products"]
    exclusive = gap_data["exclusive_by_branch"]

    print(f"  Total Catalog Products : {len(all_cat)}")
    print(f"  Common to ALL Branches : {len(common_all)}")
    print(f"    -> {', '.join(sorted(common_all)) if common_all else 'None'}")
    print()

    print("Branch Catalog Coverage:")
    for b, p_set in sorted(branch_sets.items()):
        ex = exclusive.get(b, set())
        ex_str = f"Exclusive: {', '.join(sorted(ex))}" if ex else "No exclusive products"
        print(f"  Branch {b}: {len(p_set)} product lines | {ex_str}")

    print("\nPairwise Set Comparisons:")
    for comp in gap_data["pairwise_differences"]:
        b1, b2 = comp["branches"]
        only_1 = comp["only_in_first"]
        only_2 = comp["only_in_second"]
        only_1_str = ", ".join(sorted(only_1)) if only_1 else "None"
        only_2_str = ", ".join(sorted(only_2)) if only_2 else "None"
        print(f"  * In Branch {b1} but NOT in {b2} : {only_1_str}")
        print(f"  * In Branch {b2} but NOT in {b1} : {only_2_str}")


def display_bill_statistics(df: pd.DataFrame):
    """Display NumPy bill statistics (PRD F8)."""
    display_header("Invoice Bill Statistics (NumPy)", "Distribution of Transaction Values")
    stats = get_bill_statistics(df)

    headers = ["Metric", "Value"]
    rows = [
        ["Total Invoices Analyzed", f"{stats['count']:,}"],
        ["Mean Bill Amount", f"${stats['mean']:,.2f}"],
        ["Median Bill Amount", f"${stats['median']:,.2f}"],
        ["Standard Deviation", f"${stats['std_dev']:,.2f}"],
        ["90th Percentile Bill", f"${stats['percentile_90']:,.2f}"],
        ["Minimum Invoice Total", f"${stats['min']:,.2f}"],
        ["Maximum Invoice Total", f"${stats['max']:,.2f}"],
    ]
    print(format_table(headers, rows))


def display_search_results(results_df: pd.DataFrame, query: str):
    """Display matching transaction search results (PRD F9)."""
    display_header("Search Results", f"Keyword Query: '{query}'")
    match_count = len(results_df)
    print(f"  Found {match_count:,} matching transaction(s).")
    print()

    if results_df.empty:
        print("  No records matched your search query. Try another keyword.")
        return

    display_df = results_df.head(15)
    headers = ["Invoice ID", "Branch", "Product Line", "Total", "Date", "Payment", "Rating"]
    rows = []
    for _, r in display_df.iterrows():
        rating_str = f"{r['Rating']:.1f}" if pd.notnull(r.get("Rating")) else "N/A"
        rows.append([
            r.get("Invoice ID", ""),
            r.get("Branch", ""),
            r.get("Product line", ""),
            f"${r.get('Total', 0.0):,.2f}",
            str(r.get("Date", ""))[:10],
            r.get("Payment", ""),
            rating_str,
        ])
    print(format_table(headers, rows))

    if match_count > 15:
        print(f"  ... and {match_count - 15} more records (showing top 15).")


def display_bar_charts(df: pd.DataFrame):
    """Render full suite of ASCII text bar charts (PRD F10)."""
    display_header("Text Bar Charts", "Visual Performance Without External Plotting")

    prod_df = get_revenue_by_product_line(df)
    prod_items = [(str(r["Product line"]), float(r["Revenue"])) for _, r in prod_df.iterrows()]
    print(render_bar_chart("Revenue by Product Line ($)", prod_items, unit="$"))
    print()

    branch_df = get_revenue_by_branch(df)
    branch_items = [(f"Branch {r['Branch']}", float(r["Revenue"])) for _, r in branch_df.iterrows()]
    print(render_bar_chart("Revenue by Branch ($)", branch_items, unit="$"))
    print()

    month_df = get_revenue_by_month(df)
    month_items = [(str(r["Month"]), float(r["Revenue"])) for _, r in month_df.iterrows()]
    print(render_bar_chart("Monthly Revenue ($)", month_items, unit="$"))


def prepare_monthly_report_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate comprehensive aggregated dataset for monthly_report.csv export (PRD F11).

    Parameters:
        df (pd.DataFrame): Sales DataFrame.

    Returns:
        pd.DataFrame: Aggregated monthly performance dataset.
    """
    if df.empty or "Month" not in df.columns:
        return pd.DataFrame()

    grouped = df.groupby(["Month", "Branch"], as_index=False).agg(
        Revenue=("Total", "sum"),
        Units_Sold=("Quantity", "sum"),
        Transactions=("Invoice ID", "count"),
        Avg_Bill=("Total", "mean"),
        Gross_Income=("gross income", "sum"),
    )
    grouped["Revenue"] = grouped["Revenue"].round(2)
    grouped["Avg_Bill"] = grouped["Avg_Bill"].round(2)
    grouped["Gross_Income"] = grouped["Gross_Income"].round(2)
    return grouped


def build_full_text_report(df: pd.DataFrame, low_sales_limit: float = DEFAULT_LOW_SALES_LIMIT) -> str:
    """
    Build complete comprehensive executive plain-text report for report.txt (PRD F11).

    Parameters:
        df (pd.DataFrame): Sales DataFrame.
        low_sales_limit (float): Low sales alert threshold.

    Returns:
        str: Multi-line plain-text report string.
    """
    tot_rev = get_total_revenue(df)
    tot_units = get_total_units(df)
    b_df = get_revenue_by_branch(df)
    p_df = get_revenue_by_product_line(df)
    pay = get_payment_share(df)
    stats = get_bill_statistics(df)
    rankings = get_best_and_slowest_products(df)
    alert_info = get_low_sales_alerts(df, low_sales_limit)

    sections = [
        "==================================================================",
        "          SUPERMARKET SALES & STOCK INSIGHT REPORT                ",
        "==================================================================",
        f"Transactions Analyzed          : {len(df):,}",
        f"Total Revenue Generated        : ${tot_rev:,.2f}",
        f"Total Units Sold               : {tot_units:,}",
        f"Average Bill Amount            : ${stats['mean']:,.2f}",
        "------------------------------------------------------------------",
        "",
        "1. PERFORMANCE BY BRANCH",
        "------------------------",
    ]

    for _, r in b_df.iterrows():
        sections.append(
            f"  Branch {r['Branch']}: ${r['Revenue']:,.2f} | {r['Quantity']:,} units | {r['Transactions']:,} txs | Avg Bill: ${r['Avg_Bill']:,.2f}"
        )

    sections.extend([
        "",
        "2. PERFORMANCE BY PRODUCT LINE",
        "------------------------------",
    ])
    for _, r in p_df.iterrows():
        sections.append(
            f"  {r['Product line']:<25}: ${r['Revenue']:>10,.2f} | {r['Quantity']:>5,} units | Avg Price: ${r['Avg_Price']:,.2f}"
        )

    sections.extend([
        "",
        "3. BEST & SLOWEST PERFORMERS",
        "----------------------------",
        f"  Best by Revenue    : {rankings['best_by_revenue']['product_line']} (${rankings['best_by_revenue']['revenue']:,.2f})",
        f"  Slowest by Revenue : {rankings['slowest_by_revenue']['product_line']} (${rankings['slowest_by_revenue']['revenue']:,.2f})",
        f"  Best by Units Sold : {rankings['best_by_units']['product_line']} ({rankings['best_by_units']['units']:,} units)",
        f"  Fewest Units Sold  : {rankings['slowest_by_units']['product_line']} ({rankings['slowest_by_units']['units']:,} units)",
        "",
        "4. PAYMENT METHOD DISTRIBUTION",
        "------------------------------",
    ])
    for meth, pct in pay["revenue_share_pct"].items():
        sections.append(f"  {meth:<18}: {pct:>5.2f}% of Revenue")

    sections.extend([
        "",
        "5. NUMPY BILL STATISTICS",
        "------------------------",
        f"  Mean Bill          : ${stats['mean']:,.2f}",
        f"  Median Bill        : ${stats['median']:,.2f}",
        f"  Std Deviation      : ${stats['std_dev']:,.2f}",
        f"  90th Percentile    : ${stats['percentile_90']:,.2f}",
        f"  Min Bill           : ${stats['min']:,.2f}",
        f"  Max Bill           : ${stats['max']:,.2f}",
        "",
        f"6. LOW-SALES VELOCITY ALERTS (Threshold: ${low_sales_limit:,.2f})",
        "----------------------------------------------------------",
    ])

    if not alert_info["alerts"]:
        sections.append(f"  No products below the ${low_sales_limit:,.2f} threshold.")
    else:
        for itm in alert_info["alerts"]:
            sections.append(
                f"  [ALERT] {itm['product_line']}: ${itm['revenue']:,.2f} (Deficit: ${itm['deficit']:,.2f}) -> {itm['recommendation']}"
            )

    sections.extend([
        "",
        "==================================================================",
        "End of Report",
        "==================================================================",
    ])

    return "\n".join(sections)


def export_all_reports(
    df: pd.DataFrame,
    low_sales_limit: float = DEFAULT_LOW_SALES_LIMIT,
    csv_path: str = MONTHLY_REPORT_CSV,
    txt_path: str = REPORT_TXT,
) -> tuple:
    """
    Execute exports of monthly_report.csv and report.txt (PRD F11).

    Parameters:
        df (pd.DataFrame): Sales DataFrame.
        low_sales_limit (float): Low sales threshold.
        csv_path (str): CSV export target path.
        txt_path (str): Text report export target path.

    Returns:
        tuple: (csv_success: bool, txt_success: bool)
    """
    monthly_df = prepare_monthly_report_df(df)
    csv_ok = save_csv(monthly_df, csv_path)

    full_txt = build_full_text_report(df, low_sales_limit)
    txt_ok = save_text_report(full_txt, txt_path)

    return csv_ok, txt_ok


print("Report layer loaded.")

## 🎛️ Step 7: Main Application  *(`main.py`)*
Menu handlers, dictionary dispatcher and the `main()` loop. Defining them here does not start the menu - that happens in Step 20.

In [ ]:
# =====================================================================
# STEP 7 - MAIN APPLICATION (MENU LOOP)                 (main.py)
# Menu loop and CLI routing.
# Syllabus topics used: dictionaries, functions, lists, strings, variables, operators.
# Layer rule: Calls the report and data layers. Contains NO analytical business logic.
# Only the report layer and this menu layer perform print and input operations.
# =====================================================================


def print_banner():
    """Display startup application banner."""
    print("=" * 68)
    print("   SUPERMARKET SALES & STOCK INSIGHT TOOL (CONSOLE SYSTEM)")
    print("   Fast, Syllabus-Compliant Retail Decision Analytics Engine")
    print("=" * 68)


def print_menu(current_limit: float, record_count: int, source_label: str = ""):
    """
    Print the numbered interactive console menu (PRD F12).
    Uses lists and string formatting.
    """
    menu_items = [
        "1.  Data Health Check (Duplicates & Missing Values)",
        "2.  Revenue Views (By Branch, Product Line, Month, Hour)",
        "3.  Best & Slowest Products (By Revenue & Units)",
        "4.  Payment Method Share (Volume & Percentage)",
        f"5.  Low-Sales Velocity Alerts (Threshold: ${current_limit:,.2f})",
        "6.  Branch Product Gap Analysis (Set Theory Intersection & Diffs)",
        "7.  Bill Statistics (NumPy Mean, Median, StdDev, 90th Pct)",
        "8.  Search Transactions (Case & Space Tolerant Lookup)",
        "9.  Visual Text Bar Charts (Pure String Rendering)",
        "10. Export Reports (monthly_report.csv & report.txt)",
        "11. Adjust Low-Sales Target Threshold",
        "12. Switch Dataset (Production CSV, Demo Data, or Custom File)",
        "0.  Exit System",
    ]

    print("\n" + "-" * 68)
    print(f"  MAIN MENU  [Active Dataset: {record_count:,} records | Alert Target: ${current_limit:,.2f}]")
    if source_label:
        print(f"  Source: {source_label}")
    print("-" * 68)
    for item in menu_items:
        print(f"  {item}")
    print("-" * 68)


# --- Menu Action Handlers (Functions mapped via Dictionary) ---

def handle_health_check(state: dict):
    """Action 1: Display dataset health and integrity stats."""
    display_health_check(state["health_stats"])


def handle_revenue_views(state: dict):
    """Action 2: Display multidimensional revenue aggregations."""
    display_revenue_views(state["df"])


def handle_product_rankings(state: dict):
    """Action 3: Display top and bottom performing product lines."""
    display_product_rankings(state["df"])


def handle_payment_share(state: dict):
    """Action 4: Display payment method breakdown."""
    display_payment_share(state["df"])


def handle_low_sales_alerts(state: dict):
    """Action 5: Check and display low-sales velocity alerts."""
    display_low_sales_alerts(state["df"], state["low_sales_limit"])


def handle_branch_gap(state: dict):
    """Action 6: Perform and display set difference and intersection."""
    display_branch_gap(state["df"])


def handle_bill_statistics(state: dict):
    """Action 7: Display NumPy invoice statistics."""
    display_bill_statistics(state["df"])


def handle_search(state: dict):
    """Action 8: Search transactions with case and space tolerance."""
    display_header("Search Transactions", "Case & Space Tolerant Keyword Query")
    query = input("  Enter keyword to search (product, branch, invoice, date, city): ").strip()
    if not query:
        print("  [Notice] Search query was empty. Returning to menu.")
        return

    results = search_records(state["df"], query)
    display_search_results(results, query)


def handle_bar_charts(state: dict):
    """Action 9: Render ASCII visual text bar charts."""
    display_bar_charts(state["df"])


def handle_exports(state: dict):
    """Action 10: Export monthly_report.csv and report.txt."""
    display_header("Export Reports", "Saving Deliverables to data/output/")
    csv_ok, txt_ok = export_all_reports(
        state["df"],
        state["low_sales_limit"],
        MONTHLY_REPORT_CSV,
        REPORT_TXT,
    )

    if csv_ok and txt_ok:
        print(f"  [SUCCESS] Monthly Report exported to : {MONTHLY_REPORT_CSV}")
        print(f"  [SUCCESS] Executive Text Report saved: {REPORT_TXT}")
        if state.get("is_demo"):
            print("  [Notice] These reports were built from the synthetic DEMO dataset.")
    else:
        print("  [ERROR] An error occurred while writing export files.")


def handle_adjust_threshold(state: dict):
    """Action 11: Modify the low-sales limit at runtime (PRD F6)."""
    display_header("Configure Alert Threshold", f"Current Target: ${state['low_sales_limit']:,.2f}")
    new_val_str = input("  Enter new low-sales threshold limit in dollars: ").strip()

    try:
        new_val = float(new_val_str)
        if new_val < 0:
            print("  [Error] Threshold cannot be negative.")
        else:
            state["low_sales_limit"] = new_val
            print(f"  [Updated] New low-sales alert threshold set to: ${new_val:,.2f}")
    except ValueError:
        print("  [Error] Invalid number. Target threshold remained unchanged.")


def handle_switch_dataset(state: dict):
    """Action 12: Switch between the production CSV, the demo dataset, or a custom file."""
    display_header("Dataset Switcher", "Load Alternate Sales Data")
    print(f"  Current Source: {state['active_file']}")
    print(f"  1. Main Production Dataset ({DEFAULT_DATA_FILE})")
    print("  2. Built-in Demo Dataset   (synthetic sample data for testing)")
    print("  3. Custom file path")

    choice = input("  Select dataset (1, 2, or 3): ").strip()

    try:
        if choice == "1":
            source, raw_df, is_demo = DEFAULT_DATA_FILE, load_data(DEFAULT_DATA_FILE), False
        elif choice == "2":
            source, raw_df, is_demo = DEMO_SOURCE_LABEL, generate_demo_data(), True
        elif choice == "3":
            path = input("  Enter full path to CSV file: ").strip()
            if not path:
                print("  [Notice] No path entered. Dataset unchanged.")
                return
            source, raw_df, is_demo = path, load_data(path), False
        else:
            print("  [Notice] No dataset selected. Dataset unchanged.")
            return

        clean_df, health = clean_data(raw_df)
        state["df"] = clean_df
        state["health_stats"] = health
        state["active_file"] = source
        state["is_demo"] = is_demo
        print(f"  [Success] Loaded {len(clean_df):,} records from {source}")
    except Exception as e:
        print(f"  [Error] Failed to load dataset: {e}")


def init_app_state(file_path: str = None) -> dict:
    """
    Initialize application state dictionary with loaded and cleaned data.
    If the CSV file does not exist, the built-in synthetic demo dataset is used instead.

    Parameters:
        file_path (str): Initial dataset path (defaults to DEFAULT_DATA_FILE).

    Returns:
        dict: Application state holding df, health_stats, low_sales_limit,
              active_file and is_demo.
    """
    if file_path is None:
        file_path = DEFAULT_DATA_FILE

    if os.path.exists(file_path):
        raw_df = load_data(file_path)
        source, is_demo = file_path, False
    else:
        raw_df = generate_demo_data()
        source, is_demo = DEMO_SOURCE_LABEL, True

    clean_df, health = clean_data(raw_df)

    return {
        "df": clean_df,
        "health_stats": health,
        "low_sales_limit": DEFAULT_LOW_SALES_LIMIT,
        "active_file": source,
        "is_demo": is_demo,
    }


def main():
    """
    Main application entry point.
    Runs interactive numbered menu loop using a dictionary dispatcher (PRD F12).
    """
    print_banner()

    try:
        state = init_app_state(DEFAULT_DATA_FILE)
        print(f"  [OK] Dataset loaded successfully ({len(state['df']):,} clean records).")
        if state["is_demo"]:
            print("  [Notice] supermarket_sales.csv was not found - using the synthetic DEMO dataset.")
            print("           Use option 12 to load your own CSV file.")
    except Exception as err:
        print(f"  [Startup Error] Could not load dataset: {err}")
        return

    # Dictionary mapping choice string to handler function (PRD requirement)
    menu_dispatch = {
        "1": handle_health_check,
        "2": handle_revenue_views,
        "3": handle_product_rankings,
        "4": handle_payment_share,
        "5": handle_low_sales_alerts,
        "6": handle_branch_gap,
        "7": handle_bill_statistics,
        "8": handle_search,
        "9": handle_bar_charts,
        "10": handle_exports,
        "11": handle_adjust_threshold,
        "12": handle_switch_dataset,
    }

    # Main menu loop (PRD F12)
    while True:
        try:
            print_menu(state["low_sales_limit"], len(state["df"]), state["active_file"])
            choice = input("  Select an option (0-12): ").strip()

            if choice in ("0", "exit", "quit", "q"):
                print("\n  Thank you for using Supermarket Sales & Stock Insight Tool. Goodbye!\n")
                break

            handler = menu_dispatch.get(choice)
            if handler:
                handler(state)
            else:
                print(f"\n  [Invalid Choice] '{choice}' is not a valid menu option. Please enter 0-12.")

            # Pause briefly for user to read output before showing menu again
            input("\n  Press [Enter] to return to menu...")
        except (KeyboardInterrupt, EOFError):
            # Stop button in Jupyter/Colab (or closed input) ends the session cleanly
            print("\n\n  Session interrupted. Goodbye!\n")
            break


print("Main application loaded.")

---
## Part B - Run it
## 📂 Step 8: Locate the Dataset
Looks for `supermarket_sales.csv`; offers an upload box in Colab; otherwise falls back to the built-in demo dataset.

In [ ]:
# =====================================================================
# STEP 8 - LOCATE THE DATASET
# Looks for supermarket_sales.csv in the usual places. In Google Colab it
# offers an upload box. If nothing is found, the built-in synthetic DEMO
# dataset is used so the app still runs.
# =====================================================================

candidate_paths = [
    DEFAULT_DATA_FILE,
    "supermarket_insight/data/supermarket_sales.csv",
    "data/supermarket_sales.csv",
    "supermarket_sales.csv",
    "/content/supermarket_sales.csv",
    "/content/supermarket_insight/data/supermarket_sales.csv",
]

data_file = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_file is None:
    print("supermarket_sales.csv was not found next to this notebook.")
    try:
        from google.colab import files  # only exists inside Google Colab

        print("Please upload supermarket_sales.csv (or press Cancel to use the demo data):")
        uploaded = files.upload()
        data_file = next((name for name in uploaded if name.lower().endswith(".csv")), None)
    except ImportError:
        print("Not running in Colab. Put supermarket_sales.csv in the same folder as this notebook")
        print("(or in a 'data' sub-folder) and re-run this cell.")

if data_file:
    DEFAULT_DATA_FILE = data_file
    print(f"Dataset located at: {data_file}")
else:
    print("-> Using the BUILT-IN DEMO DATASET (synthetic sample data, NOT real sales).")

## 🧹 Step 9: Data Ingestion, Cleaning & Health Check (PRD F1 & F2)
- Strips whitespace and standardizes title casing
- Drops duplicate invoice IDs
- Derives `Month` and `Hour`
- Reports missing values without crashing

In [ ]:
state = init_app_state(DEFAULT_DATA_FILE)
clean_df = state["df"]
health_stats = state["health_stats"]

print(f"Data source: {state['active_file']}")
if state["is_demo"]:
    print("NOTE: this is the synthetic DEMO dataset, not real sales data.")

display_health_check(health_stats)

print(f"\nCleaned DataFrame preview ({len(clean_df):,} records, {clean_df.shape[1]} columns):")
clean_df[["Invoice ID", "Branch", "City", "Product line", "Quantity", "Total", "Month", "Hour"]].head(5)

## 📊 Step 10: Multi-Dimensional Revenue Views (PRD F3)
By **Branch**, **Product Line**, **Month** and **Hour of day**.

In [ ]:
display_revenue_views(clean_df)

## 🏆 Step 11: Best & Slowest Performing Products (PRD F4)
Ranked by revenue and by units sold.

In [ ]:
display_product_rankings(clean_df)

## 💳 Step 12: Payment Method Share (PRD F5)

In [ ]:
display_payment_share(clean_df)

## ⚠️ Step 13: Low-Sales Velocity Alerts (PRD F6)
Change `ALERT_TARGET` to any dollar amount.

In [ ]:
# Configurable threshold (the app's default is $5,000; $52,000 makes alerts appear on the classic supermarket dataset)
ALERT_TARGET = 52000.0

display_low_sales_alerts(clean_df, limit=ALERT_TARGET)

## 🔄 Step 14: Branch Product Gap Analysis - Set Theory (PRD F7)
Intersection = products in every branch; difference = products one branch has and another lacks.

In [ ]:
display_branch_gap(clean_df)

## 📈 Step 15: Bill Statistics - NumPy (PRD F8)
Mean, median, standard deviation, 90th percentile, min and max.

In [ ]:
display_bill_statistics(clean_df)

## 🔎 Step 16: Tolerant Search (PRD F9)
Case-insensitive, whitespace-tolerant matching across invoice, branch, city, product line, payment, date and month.

In [ ]:
# Try: 'health and beauty', 'credit card', 'branch a', 'electronic'
query_string = "  health   and   BEAUTY  "
results = search_records(clean_df, query_string)

display_search_results(results, query_string)

## 📊 Step 17: Visual Text Bar Charts (PRD F10)
ASCII bars built purely with string multiplication.

In [ ]:
display_bar_charts(clean_df)

## 💾 Step 18: Export Deliverables (PRD F11)
Saves `monthly_report.csv` and `report.txt` into `data/output/` (and offers a download in Colab).

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_ok, txt_ok = export_all_reports(
    clean_df,
    low_sales_limit=ALERT_TARGET,
    csv_path=MONTHLY_REPORT_CSV,
    txt_path=REPORT_TXT,
)

print(f"✅ Monthly aggregated CSV exported: {MONTHLY_REPORT_CSV} (Success: {csv_ok})")
print(f"✅ Executive text summary exported : {REPORT_TXT} (Success: {txt_ok})")

# If in Google Colab, offer download triggers
try:
    from google.colab import files
    print("Downloading files to your local machine...")
    files.download(MONTHLY_REPORT_CSV)
    files.download(REPORT_TXT)
except Exception:
    print("Files are saved in your workspace at data/output/")

## 🧪 Step 19: Self-Check Tests
The original `tests/run_tests.py` was not part of the uploaded files, so this cell contains a fresh set of `assert`-based checks that run on a tiny built-in dataset with known answers.

In [ ]:
# =====================================================================
# STEP 19 - SELF-CHECK TESTS (plain assert statements)
# Runs against a tiny in-memory dataset with known answers, so it never
# depends on your CSV. Nothing here prints into the app's own output.
# =====================================================================

import io
import tempfile
from contextlib import redirect_stdout


def _approx(a, b, tol=1e-6):
    return abs(a - b) < tol


def _fixture_clean():
    """Four unique invoices (one duplicated, messy text, one missing rating)."""
    cols = [
        "Invoice ID", "Branch", "City", "Customer type", "Gender", "Product line",
        "Unit price", "Quantity", "Tax 5%", "Total", "Date", "Time", "Payment",
        "cogs", "gross margin percentage", "gross income", "Rating",
    ]
    r1 = ["T-001", "A", "Yangon", "Member", "Female", "  health and beauty ", 10.0, 2, 1.0, 21.0,
          "1/5/2019", "13:08", "Ewallet", 20.0, 4.761904762, 1.0, 9.1]
    r2 = ["T-002", " a ", "Yangon", "Normal", "Male", "Food and beverages", 50.0, 2, 5.0, 105.0,
          "1/9/2019", "10:30", "Cash", 100.0, 4.761904762, 5.0, 7.0]
    r3 = ["T-003", "B", "Mandalay", "Member", "Male", "FOOD AND BEVERAGES", 20.0, 5, 5.0, 105.0,
          "2/3/2019", "19:45", "Credit card", 100.0, 4.761904762, 5.0, np.nan]
    r4 = ["T-004", "C", "Naypyitaw", "Normal", "Female", "Sports and travel", 30.0, 1, 1.5, 31.5,
          "3/8/2019", "14:00", "Ewallet", 30.0, 4.761904762, 1.5, 8.0]
    raw = pd.DataFrame([r1, r2, r3, list(r3), r4], columns=cols)  # r3 appears twice
    return clean_data(raw)


def test_cleaning_removes_duplicates_and_normalizes_text():
    df, health = _fixture_clean()
    assert len(df) == 4 and health["duplicates_removed"] == 1
    assert set(df["Product line"]) == {"Health And Beauty", "Food And Beverages", "Sports And Travel"}
    assert set(df["Branch"]) == {"A", "B", "C"}


def test_month_and_hour_are_derived():
    df, _ = _fixture_clean()
    assert set(df["Month"]) == {"January", "February", "March"}
    assert int(df.loc[df["Invoice ID"] == "T-001", "Hour"].iloc[0]) == 13


def test_health_check_counts_missing_values():
    _, health = _fixture_clean()
    assert health["missing_values"]["Rating"] == 1
    assert health["total_missing"] == 1


def test_revenue_totals_and_branch_view():
    df, _ = _fixture_clean()
    assert _approx(get_total_revenue(df), 262.5) and get_total_units(df) == 10
    by_branch = get_revenue_by_branch(df)
    assert by_branch.iloc[0]["Branch"] == "A" and _approx(by_branch.iloc[0]["Revenue"], 126.0)


def test_product_rankings():
    df, _ = _fixture_clean()
    r = get_best_and_slowest_products(df)
    assert r["best_by_revenue"]["product_line"] == "Food And Beverages"
    assert r["slowest_by_revenue"]["product_line"] == "Health And Beauty"
    assert r["best_by_units"]["units"] == 7
    assert r["slowest_by_units"]["product_line"] == "Sports And Travel"


def test_payment_share():
    df, _ = _fixture_clean()
    p = get_payment_share(df)
    assert _approx(p["count_share_pct"]["Ewallet"], 50.0)
    assert _approx(p["revenue_share_pct"]["Ewallet"], 20.0)
    assert _approx(sum(p["revenue_share_pct"].values()), 100.0)


def test_low_sales_alerts():
    df, _ = _fixture_clean()
    a = get_low_sales_alerts(df, 100.0)
    assert a["alert_count"] == 2 and a["total_products_checked"] == 3
    assert a["alerts"][0]["product_line"] == "Health And Beauty"  # biggest deficit first
    assert a["safe_products"][0]["product_line"] == "Food And Beverages"


def test_branch_gap_set_operations():
    df, _ = _fixture_clean()
    g = get_branch_product_gap(df)
    assert len(g["all_catalog_products"]) == 3 and g["common_all_branches"] == set()
    assert g["exclusive_by_branch"]["A"] == {"Health And Beauty"}
    assert g["exclusive_by_branch"]["B"] == set()
    assert len(g["pairwise_differences"]) == 3


def test_bill_statistics_numpy():
    df, _ = _fixture_clean()
    s = get_bill_statistics(df)
    assert s["count"] == 4 and _approx(s["mean"], 65.625) and _approx(s["median"], 68.25)
    assert _approx(s["min"], 21.0) and _approx(s["max"], 105.0)


def test_rating_averages_ignore_missing_values():
    df, _ = _fixture_clean()
    assert _approx(get_rating_averages(df)["overall_avg"], 8.03)


def test_search_is_case_and_space_tolerant():
    df, _ = _fixture_clean()
    assert len(search_records(df, "  FOOD    and   beverages ")) == 2
    assert len(search_records(df, "mandalay")) == 1
    assert get_transaction_by_invoice(df, " t-004 ")["Branch"] == "C"
    assert len(filter_records(df, branch="a", min_total=50)) == 1


def test_empty_dataframe_is_handled_safely():
    empty = pd.DataFrame()
    assert get_total_revenue(empty) == 0.0
    assert get_bill_statistics(empty)["count"] == 0
    assert get_branch_product_gap(empty)["pairwise_differences"] == []
    assert get_low_sales_alerts(empty)["alert_count"] == 0
    assert "No data" in render_bar_chart("Empty", [])


def test_text_bar_and_table_helpers():
    assert build_text_bar(50, 100, width=10) == "[#####     ]"
    assert build_text_bar(0, 100, width=10) == "[          ]"
    table_lines = format_table(["Name", "Revenue"], [["A", "$1.00"], ["Longer", "$22.00"]]).split("\n")
    assert len({len(line) for line in table_lines}) == 1  # every line is the same width


def test_every_display_screen_runs():
    df, health = _fixture_clean()
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        display_health_check(health)
        display_revenue_views(df)
        display_product_rankings(df)
        display_payment_share(df)
        display_low_sales_alerts(df, 100.0)
        display_branch_gap(df)
        display_bill_statistics(df)
        display_search_results(search_records(df, "food"), "food")
        display_bar_charts(df)
    assert "Revenue Views".upper() in buffer.getvalue().upper()


def test_exports_write_both_files():
    df, _ = _fixture_clean()
    with tempfile.TemporaryDirectory() as tmp:
        csv_path = os.path.join(tmp, "out", "monthly_report.csv")
        txt_path = os.path.join(tmp, "out", "report.txt")
        assert export_all_reports(df, 100.0, csv_path, txt_path) == (True, True)
        assert len(pd.read_csv(csv_path)) > 0
        with open(txt_path, encoding="utf-8") as f:
            assert "SUPERMARKET SALES & STOCK INSIGHT REPORT" in f.read()


def run_all_tests() -> bool:
    """Run every test, print PASS/FAIL per test, and return True only if all passed."""
    tests = [
        test_cleaning_removes_duplicates_and_normalizes_text,
        test_month_and_hour_are_derived,
        test_health_check_counts_missing_values,
        test_revenue_totals_and_branch_view,
        test_product_rankings,
        test_payment_share,
        test_low_sales_alerts,
        test_branch_gap_set_operations,
        test_bill_statistics_numpy,
        test_rating_averages_ignore_missing_values,
        test_search_is_case_and_space_tolerant,
        test_empty_dataframe_is_handled_safely,
        test_text_bar_and_table_helpers,
        test_every_display_screen_runs,
        test_exports_write_both_files,
    ]
    passed = 0
    for test in tests:
        try:
            test()
            print(f"  [PASS] {test.__name__}")
            passed += 1
        except Exception as err:  # AssertionError or an unexpected crash
            print(f"  [FAIL] {test.__name__}: {type(err).__name__}: {err}")
    print(f"\n{passed}/{len(tests)} tests passed.")
    return passed == len(tests)


test_success = run_all_tests()
assert test_success, "One or more automated tests failed!"

## 🎮 Step 20: Interactive Menu
Launches the full numbered console menu. Type a number + Enter. **Type `0` to exit.** (If you press the Stop button, the loop also ends cleanly.)

In [ ]:
# Launch the interactive menu (type 0 to exit)
main()